# 8.5 n-gram language models

An n-gram model is language modeling before neural networks: predict the next token by trusting the counts that followed the same short history before.

Conditional probability estimates the next token from a short history. Add-alpha smoothing prevents one unseen transition from making a whole sequence impossible.

Save a copy to Drive to edit.

In [ ]:

import math
import random
import re
import unicodedata
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8)
np.random.seed(8)


def exact_accuracy(predictions, expected):
    correct = 0
    for pred, gold in zip(predictions, expected):
        if pred == gold:
            correct += 1
    return correct / max(1, len(expected))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def tokenize_doc(doc):
    return re.findall(r"[a-z0-9]+", doc.casefold())


def show_matrix(ax, matrix, title, xlabels=None, ylabels=None):
    image = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_title(title)
    if xlabels is not None:
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, rotation=45, ha="right")
    if ylabels is not None:
        ax.set_yticks(range(len(ylabels)))
        ax.set_yticklabels(ylabels)
    return image


## The concept, built once (D1): add-$\alpha$ n-gram probabilities

For a bigram model over vocabulary $V$, $$P(w\mid h)=\frac{c(h,w)+\alpha}{\sum_{v\in V}c(h,v)+\alpha |V|}.$$ In the lesson corpus `a b a b a c`, unigram counts are $[3,2,1]$ over six tokens and the continuation row after `a` is $[0,2,1]$.

In [ ]:

def ngram_lm(tokens, n=2, alpha=0.0, vocabulary=None):
    if vocabulary is None:
        vocabulary = sorted(set(tokens))
    histories = defaultdict(Counter)
    if n == 1:
        histories[()].update(tokens)
    else:
        for i in range(len(tokens) - n + 1):
            history = tuple(tokens[i:i + n - 1])
            word = tokens[i + n - 1]
            histories[history][word] += 1
    return {"n": n, "alpha": alpha, "vocab": vocabulary, "histories": histories}


def next_prob(model, history, word):
    history = tuple(history)
    counts = model["histories"].get(history, Counter())
    vocab = model["vocab"]
    alpha = model["alpha"]
    numerator = counts.get(word, 0) + alpha
    denominator = sum(counts.get(v, 0) for v in vocab) + alpha * len(vocab)
    if denominator == 0:
        return 0.0
    return numerator / denominator

lesson_tokens = "a b a b a c".split()
lesson_vocab = sorted(set(lesson_tokens))
unigram_counts = [lesson_tokens.count(token) for token in lesson_vocab]
bigram_raw = ngram_lm(lesson_tokens, n=2, alpha=0.0, vocabulary=lesson_vocab)
row_after_a = [bigram_raw["histories"][("a",)].get(token, 0) for token in lesson_vocab]
raw_probs = [next_prob(bigram_raw, ["a"], token) for token in lesson_vocab]

assert lesson_vocab == ["a", "b", "c"]
assert unigram_counts == [3, 2, 1]
assert sum(unigram_counts) == 6
assert lesson_tokens.count("a") / len(lesson_tokens) == 0.5
assert row_after_a == [0, 2, 1]
assert np.allclose(raw_probs, [0.0, 2 / 3, 1 / 3])


The same method scores full sequences by multiplying conditional probabilities, reports perplexity, and can use add-one smoothing for unseen continuations.

In [ ]:

def sequence_log_prob(model, tokens):
    n = model["n"]
    log_prob = 0.0
    count = 0
    for i in range(n - 1, len(tokens)):
        history = tokens[i - n + 1:i]
        word = tokens[i]
        prob = next_prob(model, history, word)
        if prob <= 0:
            return float("-inf"), count + 1
        log_prob += math.log(prob)
        count += 1
    return log_prob, count


def perplexity(model, tokens):
    log_prob, count = sequence_log_prob(model, tokens)
    if count == 0:
        return float("inf")
    if log_prob == float("-inf"):
        return float("inf")
    return math.exp(-log_prob / count)

bigram_smooth = ngram_lm(lesson_tokens, n=2, alpha=1.0, vocabulary=lesson_vocab)
smoothed_probs = [next_prob(bigram_smooth, ["a"], token) for token in lesson_vocab]

assert np.allclose(smoothed_probs, [1 / 6, 3 / 6, 2 / 6])


## Dataset ladder: inline sequences from the six-token toy to domain-shifted text

In [ ]:

lm_ladder = [
    {
        "name": "D1 lesson six-token corpus",
        "train": "a b a b a c".split(),
        "eval": "a b a c".split(),
    },
    {
        "name": "D2 five clean short sequences",
        "train": "start ad click end start ad view end start bid click end start bid view end".split(),
        "eval": "start ad click end start bid view end".split(),
    },
    {
        "name": "D3 unseen transitions and sparse rows",
        "train": "user likes coffee user likes tea user buys coffee user buys book".split(),
        "eval": "user buys tea user likes book".split(),
    },
    {
        "name": "D4 tiny inline headlines",
        "train": "market rallies after earnings team wins final storm delays flights market falls after forecast team signs sponsor".split(),
        "eval": "market rallies after forecast team wins sponsor".split(),
    },
    {
        "name": "D5 longer domain-shifted sequences",
        "train": "campaign manager predicts click growth invoice system sends receipt creator uploads video campaign budget changes creative asset wins click".split(),
        "eval": "creator sends invoice campaign predicts video growth budget wins receipt".split(),
    },
]

for rung in lm_ladder:
    print(rung["name"], "train tokens", len(rung["train"]), "eval tokens", len(rung["eval"]))
    print("sample", rung["train"][:6])


In [ ]:

lm_results = []
for index, rung in enumerate(lm_ladder, start=1):
    vocabulary = sorted(set(rung["train"]) | set(rung["eval"]))
    model = ngram_lm(rung["train"], n=2, alpha=1.0, vocabulary=vocabulary)
    ppl = perplexity(model, rung["eval"])
    lm_results.append({"rung": index, "name": rung["name"], "perplexity": ppl, "model": model})

for row in lm_results:
    print(row["rung"], row["name"], f"perplexity={row['perplexity']:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(lm_results):
    model = row["model"]
    vocab = model["vocab"][:8]
    histories = list(model["histories"].keys())[:8]
    matrix = np.zeros((len(histories), len(vocab)))
    for i, history in enumerate(histories):
        for j, word in enumerate(vocab):
            matrix[i, j] = next_prob(model, history, word)
    labels = [" ".join(history) if history else "<uni>" for history in histories]
    show_matrix(axes[0, idx], matrix, f"D{idx + 1} transitions", vocab, labels)

x = [row["rung"] for row in lm_results]
y = [row["perplexity"] for row in lm_results]
axes[1, 0].plot(x, y, marker="o")
axes[1, 0].set_title("Perplexity vs sequence complexity")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("perplexity")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: one unseen transition can kill the sentence

In [ ]:

d5 = lm_ladder[-1]
vocabulary = sorted(set(d5["train"]) | set(d5["eval"]))
unsmoothed = ngram_lm(d5["train"], n=2, alpha=0.0, vocabulary=vocabulary)
smoothed = ngram_lm(d5["train"], n=2, alpha=0.5, vocabulary=vocabulary)
zero_ppl = perplexity(unsmoothed, d5["eval"])
fixed_ppl = perplexity(smoothed, d5["eval"])

print("unsmoothed perplexity", zero_ppl)
print("smoothed perplexity", fixed_ppl)
print("P(invoice | sends) unsmoothed", next_prob(unsmoothed, ["sends"], "invoice"))
print("P(invoice | sends) smoothed", next_prob(smoothed, ["sends"], "invoice"))


## Evaluate it + Practice

- Metric: perplexity on held-out token sequences
- No-skill baseline: unigram probabilities that ignore the history row
- Cheap sanity check: D1 counts must be [3,2,1] and add-one after a must be [1/6,3/6,2/6]
- Ablation: set alpha to 0 and any unseen bigram can make perplexity infinite
- Failure signals: zero sentence probability, sparse long-history rows, or PPL comparisons across tokenizers

### Practice

**Practice:** Try trigram histories on D2 and compare perplexity with the bigram model.

**Practice:** Change alpha from 1.0 to 0.1 on D3 and inspect unseen-transition probabilities.